---
title: Week 4.6, Group Project Analysis of a leachate dataset
subject: Group Project ECTB1230 2026
subtitle: Example notebook for Day 1
authors:
  - name: Timo Heimovaara
    affiliations:
      - Delft University of Technology, department of Geoscience & Engineering
    orcid: 0000-0003-4230-7476
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-05-20
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Introduction

Within the Netherlands we have been carrying out a project called "Introduction of Sustainable Aftercare of Landfills" or in Dutch: Introductie Duurzaam Stortbeheer [iDS](https://duurzaamstortbeheer.nl/). The aim of this project is to investigate if we can reduce the emission potential of a wastebody by active treatment using infiltration of water aeration. Infiltration is hypothesized to increase the flushing of dissolved contaminants from the wastebody, aeration is hypothesized to simulate the biodegradation of organic matter in the waste body. Both approaches will eventually lead to lower contaminant concentrations in the leachate.

Sanitary engineered landfills are technical facilities which allow us to store waste indefinitely. In order to protect the environment and human health from emissions, these sites are fitted with impermeable bottom liners with a drainage systems for leachate collection, and gas extraction systems within the waste body for collection of methane and other greenhouse gases being produced in the waste body. After the landfilling has been completed the current regulations require the wastebody to be capped with a water tight liner so that no rainfall can infiltrate. As a result, the driving force for leachate emissions is no longer present. The drawback, however, is that this coverliner needs to be replaced every 75 years. This is nicely illustrated with the animation you can find on [iDS](https://duurzaamstortbeheer.nl/) at the bottom of the page.

Pilot projects are being carried out within the context of iDS at three landfills: Kragge near Bergen op Zoom, Braambergen near Almere and Wieringermeer near Wieringermeer. A large amount of background information can be found in the ["background"](https://duurzaamstortbeheer.nl/achtergrond/) section of the iDS website. Here you can find information on the three pilot projects in the section ["Project documents"](https://duurzaamstortbeheer.nl/projectstukken/) and a number of [publications](https://duurzaamstortbeheer.nl/publicaties/) that have been written in the course of the project. 

Leachate quality from the three pilot projects have been measured with a relatively high frequency since 2012, the start of the base-line monitoring. Preparation of the active treatment began in 2016 and the treatment was started in 2018 and will continue until 2029. 

Your task is to analyse a leachate data set in order to answer the following questions:
1. What is the likely composition of the leachate within the waste body?
2. How much solids will precipitate from the leachate once it is exposed to air in the water treatment plant?
3. How much solids will have preciptated as the leachate moved from the bulk of the waste to the water treatment system?
4. Do these processes vary over time.

In the ["Project documents"](https://duurzaamstortbeheer.nl/projectstukken/) section of the iDS website, you can find the original plans for the pilot projects. The general overview of the iDS projects is given in the ["Integraal Plan van Aanpak"](https://duurzaamstortbeheer.nl/wp-content/uploads/2023/09/IENM-BSK-2014-116919-Def-concept-IPvA-versie-mei-2014.pdf). The site specific plans can be found in the documents starting with "Deel van Aanpak". You can also find two documents in English giving similar information: "Project plan Sustainable Landfill Management...".

## Project assignment
The assignment that you need to do is to carry out an analysis of a dataset that is provided to you. In many cases where routine interpretations are done, it is most efficient to start with a pre-existing approach or script for your analysis and then modify the approach to your specific needs. This is also the approach you need to do here. The notebooks provided to you, show a similar analysis so you can apply this notebook as your template. 

Your responsibility is to adapt the notebook and the interpretation where necessary. At the same time you need to understand what you are doing. The generated results need to be included in your final report.


## Project actvities Day 1

Before you can start working on analysing the dataset, you have to familiarize yourself with the data provided to you. The data is provided in an Excel sheet. The data is an export from a single waste body from one of the pilot projects and contains the date a sample was taken and the results of a chemical analysis in the laboratory.

First you need to obtain an overview of the data. Your assignment for this day is to:
1. Import the data;
2. Get a quick over view of the content and the structure of the dataset;
3. Understand how to plot the time series in the data set, save the figures to a file and create an overview report;
4. Need to calculate molar concentrations from mg/l values;
5. Think about what questions you want to resolve with this data?
    - Saturation status of the samples as they are;
    - What were mostly likely conditions where the samples originated?
    - What will happen to the samples if the leachate would be discharged to a system at atmospheric conditions.
5. Prepare the interface to PyOrchestra, have a look at provided GUI of Orchestra and the corresponding Chemistry File

All steps you need to do have been shown earlier in weeks 3.7 and 4.2. To help you get started I have provided this notebook where I show how to carry out the first 4 steps. You need to create your own notebook in order to analyse the dataset provided to you.
To do this you create a new Notebook in Jupyter-lab. You can copy the python cells below to your own notebook. Please make sure that you adapt the cells where necessary so that it matches your own data.

The data is provided as an Excel Workbook. I assume you will use pandas in order to process the data and seaborn to plot the output. In the following text I will provide you with some hints how to process the data using pandas.

In order to import these using pandas, you need to need install openpyxl:  
- mamba install -c conda-forge openpyxl

I suggest that have the [Python Data Sicence Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) by Jake Vanderplas available in order to have a better grip of the python concepts. This book gives an introduction on NumPy, Pandas, Matplotlib (with a section on seaborn). These are probably the main packages you will be using during your time at the TU Delft. 

First we import the required python libraries.

In [1]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

# The figures will be shown in a separate window, not inline in the notebook 
%matplotlib qt 
sns.set() 


In [2]:
# This code section is required for the Jupyter Book to find the input files for the 
# Orchestra simulations. The input files are located in the same directory as this notebook, 
# but when running the notebook in Jupyter Book, the current working directory is the book root, 
# not the directory of this notebook. Therefore, we need to find the path to the book root and 
# then construct the path to the input files from there.

def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:

orchestra_path = path_from_book_root("content", "project_THe", "Orchestra_Project")
# print(orchestra_path)

# Please note: When you are running the notebook from the folder 
# with the Orchestra files use the next line:
# orchestra_path = '.'

### Dataset  

This data set is an export from the iDS database of the iDS project carried out by the Dutch Sustainable Landfill foundation.

The exported data contains the the results from laboratory analyses of 
the following parameters
[
    'pH', 'Natrium [Na]', 'Kalium [K]', 'Calcium [Ca]', 
    'Magnesium [Mg]', 'IJzer [Fe]', 'Mangaan [Mn]', 
    'Ammonium (als NH4)', 'Fosfaat (als PO4)',
    'Bicarbonaat', 'Chloride', 'Sulfide', 
    'Sulfaat (als SO4)', 
    'Temperatuur',
]

This list contain the macro parameters (where in general the concentration in the leachate is above 1 mg/l).

The data has been preprocessed in order to remove obvious outliers.

As a consequence of the measurement strategy and the outlier removement, the number of analyses on the samples taken over time varies significantly.

Measurement strategy:
- originally: 1 to 2 samples per year;
- iDS: 26 samples per year, parameters vary per frequency
    - parameters 26 times per year (bi weekly)
    - parameters 13 times per year (every four weeks)
    - parameters 4 times per year  (quarterly)
    - parameters 1 time per year (yearly)
    - parameters that are taken irregurlary

The details of the measurement frequencies can be found in the project plans on the [iDS projects](https://duurzaamstortbeheer.nl/projectstukken/) website.


We first import the data and print the some rows in order to see the data stored in the spreadsheet. You can also have a look at the spreadsheet it self using Excel or something similar. Please note, do not edit the raw data file.


In [3]:
# %% 1
# import the data set from the excel file.
# We will use the data from the leachate monitoring at PP-11N.

df_leachate = pd.read_excel('data/df_macros_PP-11N.xlsx')

# check the column names
print(f'The dataset has the following columns: {list(df_leachate.columns)}')
# check the number of rows and columns
print(f'The dataset has {df_leachate.shape[0]} rows and {df_leachate.shape[1]} columns.')

# check the first few rows of the dataset
print(df_leachate.head(10))

# check the last few rows of the dataset
print(df_leachate.tail(10))


The dataset has the following columns: ['compartment', 'measpointname', 'date', 'cname', 'val_mgl', 'uname_mgl']
The dataset has 2240 rows and 6 columns.
  compartment measpointname       date              cname   val_mgl uname_mgl
0       BB11N        PP-11N 2000-11-07                 pH     6.800         -
1       BB11N        PP-11N 2000-11-07  Sulfaat (als SO4)   780.000      mg/l
2       BB11N        PP-11N 2001-09-18                 pH     0.007      mg/l
3       BB11N        PP-11N 2001-09-18           Chloride     0.670      mg/l
4       BB11N        PP-11N 2002-03-20                 pH     7.100         -
5       BB11N        PP-11N 2002-03-20  Sulfaat (als SO4)     8.900      mg/l
6       BB11N        PP-11N 2002-03-20           Chloride   890.000      mg/l
7       BB11N        PP-11N 2003-07-15           Chloride  1700.000      mg/l
8       BB11N        PP-11N 2003-07-15                 pH     7.300         -
9       BB11N        PP-11N 2003-07-15  Sulfaat (als SO4)    21.00

This data set is a so-called list type. We can access the data using the *cname* and/or *date* column for further processing. The earliest available data are from 2000, the latest are for the end of April 2026. For most dates, multiple chemical components hvae been analysed which means that we can use the date to select a "sample" on which multiple parameters have been measured.



### Step 1: Plotting the timeseries for each chemical component in the dataset
The first thing to do is to get an overview of the data present. As you can see have 2240 rows with data points. Each row represents a laboratory analysis result from a sample with a specific date, for a specific component. We do not know which components are present and how many analyses are present for each sample.
A quick way to obtain the required overview is to generate a series of figures and look at these. 
- We create a separate folder to store these figures: ./Figures;
- We plot time series for each component in separate figures labelled by *cname*;

In [4]:
# Use seaborn line plot to plot the concentration of each component over time. 
# Use the date as x-axis and the concentration as y-axis. Use different colors 
# for different components. Add a legend to the plot.

plt.close('all')   # close all previous figures (good practice when running a new code)

# Generate a list of all unique chemical components in the dataset
component_list = df_leachate['cname'].unique()   # get the unique component names

# Prepare for plotting, it is good practice to have the figure and axis handles
# available for annotating the plots
figs = []   # initialize a counter for the figure index
axs = []   # initialize a counter for the axis index

# We will run a loop over all components in component_list, but we also
# need a counter to access the figs or axs list. We initalize this to -1 so
# after the first-update it starts with zero
ii = -1   # initialize a counter for the component index

# Loop over all components in component_list
for cn in component_list:
    ii += 1
    fig, ax = plt.subplots()  # create a new figure and axis for each component
    figs.append(fig)
    axs.append(ax)
    sns.lineplot(
        data=df_leachate[df_leachate['cname'] == cn],
        x='date',
        y='val_mgl',
        ax=axs[ii],
        marker='o',        
        )
    axs[ii].set_title(f'Concentration of {cn} over time')
    axs[ii].set_xlabel('Date') 
    axs[ii].set_ylabel('Concentration (mg/L)')

    # write figure to a file in the local Figures folder. The name is defined by cn
    fig.savefig(f'Figures/{cn}_concentration_over_time.png', dpi=300, bbox_inches='tight')


In [5]:
# We have saved the figures to the Figures folder so we can close them.
# The figures are in *.png format which you can easily import in to your report 
plt.close('all')

```{note}
**Give an intepretation of the concentration time series**
The reason for collecting samples over time is that the landfill shows a dynamic behavior which is driven by seasonal changes. One of the most notciable is the seasonal change in the leachate production which is related to the net precipitation. Net precipitation is the difference between the rainfall and the evapo-transpiration from the cover-layer of the waste body. The evapo-transpiration in summer is much larger than in the winter, it is so high on many days the evpo-transpiration exceeds rainfall by far (there are many days where it does not rain and plants still evaporate large amounts of water). As a result leachate production in summer is much smaller than in the winter.

Understanding this, can you explain the patterns you see in the leachate concentration variations over time. For this interpretation, consider the possible reactions a chemical component can undergo.
```

### Step 2: Use pyOrchestra to analyse a sample with the most parameters in the dataset

In order to carry out a geochemical analysis we need to have the following information:
- Which chemical components will we used as our master-species?
- Which samples in the data-set have largest number of chemical components analysed?

For the first question the answer is already available. In the above code we have created a list of unique components. We can print this list.

In [6]:
print(component_list)

<StringArray>
[                'pH',  'Sulfaat (als SO4)',           'Chloride',
        'Temperatuur',       'Calcium [Ca]',      'Silicium [Si]',
         'IJzer [Fe]',       'Mangaan [Mn]',         'Kalium [K]',
 'Ammonium (als NH4)',       'Natrium [Na]',        'Bicarbonaat',
            'Sulfide',     'Magnesium [Mg]',  'Fosfaat (als PO4)']
Length: 15, dtype: str


The reason for us to require a sample that has measurements of as many parameters as possible, is that we then can assess the complete chemical system for which we have data. To find the dates in the data-set with most analyses, we use some tools available in pandas. The background for these tools are clearly described in the [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html).


We use the concept of data aggregation or grouping. Our aim is to creating a new dataframe from the data containing the number of parameters per sampling date.

We use methods built in the pandas library which are associated with the pandas dataframe:
- count the number of parameters (identified in the columne *cname*) by grouping over *measurementpointname* and *date*;
- using count() as the aggregation function;
- renaming the output column to *macro_count*
- and then finally sorting the data frame using the values in *macro_count* in descending order

The first record in this data frame contains the sample, identifed by *date* with the most parameters. The top rows in this table contain the samples with most parameters, the bottom rows the samples with the least number of parameters.

We will proceed with the date with the most parameters to do the first analysis.

In [7]:
# %% 2
# Check which dates have most parameters measured.
# We will use these parameters as our input for the Orchestra 
# calculation.

# We can use the groupby function in pandasto group the data by date 
# and count the number of parameters measured for each date.
# count() creates a new column with the count of the number of parameters
# measured for each date which we will call macro_count using reset_index 
# to create a new dataframe with the macro_count column and sort the values 
# by macro_count in descending order.

df_par_counts = (
    df_leachate.groupby(['measpointname', 'date'])['cname']
    .count()
    .reset_index(name='macro_count')
    .sort_values('macro_count', ascending=False)
)

# The first value is a sample (date) with the most parameters measured
date_with_most_pars = df_par_counts.iloc[0]['date']

print(df_par_counts.head(10))

print(f"The date with the most parameters measured is: {date_with_most_pars}")


    measpointname       date  macro_count
333        PP-11N 2025-12-11           15
281        PP-11N 2023-12-12           15
291        PP-11N 2024-04-25           14
50         PP-11N 2013-05-28           14
307        PP-11N 2024-12-10           14
38         PP-11N 2012-12-11           14
239        PP-11N 2022-04-28           14
68         PP-11N 2014-01-21           13
278        PP-11N 2023-10-19           13
27         PP-11N 2012-08-21           13
The date with the most parameters measured is: 2025-12-11 00:00:00


We see that the maximum number of components in the data set is 15 for two dates, then 14 for five dates and then 13. For our first analysis we will work with one of the two data sets with 15 analysis results. 

### Calculate the concentrations in mol/l
The analysis results are given in mg/l, for the geochemical calculations with Orchestra we require the units to be in mol/l, we therefore need a conversion table to do this. At the same time, we require a translation of the component names to those used in the pyOrchestra library. We do both by creating a python dictionary.

Please note, that if your data-set has additional parameters, you need to adjust the following dataframe accordingly.


In [8]:
# componentname, molar mass (g/mol), Orchestra parameter name
conversion_table = pd.DataFrame(
    data={
        'Sulfaat (als SO4)': [96.06, 'SO4-2.tot'],
        'Sulfide': [32.07, 'S-2.tot'],
        'Natrium [Na]': [22.99, 'Na+.tot'],
        'IJzer [Fe]': [55.85, 'Fe+2.tot'],
        'Magnesium [Mg]': [24.31, 'Mg+2.tot'],
        'Calcium [Ca]': [40.08, 'Ca+2.tot'],
        'Ammonium (als NH4)': [18.04, 'NH4+.tot'],
        'Chloride': [35.45, 'Cl-.tot'],
        'Bicarbonaat': [61.02, 'HCO3-.tot'],
        'Fosfaat (als PO4)': [94.97, 'PO4-3.tot'],
        'Kalium [K]': [39.10, 'K+.tot'],
        'Silicium [Si]': [28.09, 'Si.tot'],
        'Mangaan [Mn]': [54.94, 'Mn+2.tot'],
        'Temperatuur': [1e-3, 'T'], 
        'pH': [1e-3, 'pH'], # please note the factor 1e-3 which will be corrected for in the conversion to moles/l.
    }, 
    index=['molar_mass','orchestra_parameter']).T
    


In [9]:
print(conversion_table)


                   molar_mass orchestra_parameter
Sulfaat (als SO4)       96.06           SO4-2.tot
Sulfide                 32.07             S-2.tot
Natrium [Na]            22.99             Na+.tot
IJzer [Fe]              55.85            Fe+2.tot
Magnesium [Mg]          24.31            Mg+2.tot
Calcium [Ca]            40.08            Ca+2.tot
Ammonium (als NH4)      18.04            NH4+.tot
Chloride                35.45             Cl-.tot
Bicarbonaat             61.02           HCO3-.tot
Fosfaat (als PO4)       94.97           PO4-3.tot
Kalium [K]               39.1              K+.tot
Silicium [Si]           28.09              Si.tot
Mangaan [Mn]            54.94            Mn+2.tot
Temperatuur             0.001                   T
pH                      0.001                  pH


The data provided contains information for 15 parameters of which are all present in the dataset for the two dates with most parameters.

Other samples in the data set have less parameters. For the further analysis on samples with less analysed components, we would like to include these parameters in the analysis as well. 

In order to handle the missing values we aim to use the mean value from the total dataset. We obtain the mean values from the data set with the data aggregation methods available in pandas using a similar approach as above for obtaining df_par_counts. In addition to calculating the average concentration in mg/l we also calculate the concentration in mol/l using the information present in the conversion_table dataframe.

In [10]:
# We will calculate the average concentration in mg/l for each component and add it to the conversion table.

df_comp_means = (
    df_leachate.groupby('cname')['val_mgl']
    .mean()
    .reset_index(name='avg_val_mgl')
    .set_index('cname')
)

#print(df_comp_means)

# We add the avg_val_mgl to the conversion table 
# for the components that are in the conversion table.
conversion_table = (
    conversion_table.merge(
        df_comp_means, 
        left_index=True,
        right_index=True,
        how='left')
)

# calculate the average concentration in moles/l for each component
conversion_table['avg_val_mol_l'] = conversion_table['avg_val_mgl'] / conversion_table['molar_mass'] * 1e-3

print(conversion_table)

                   molar_mass orchestra_parameter  avg_val_mgl avg_val_mol_l
Sulfaat (als SO4)       96.06           SO4-2.tot   334.525966      0.003482
Sulfide                 32.07             S-2.tot     2.070439      0.000065
Natrium [Na]            22.99             Na+.tot   589.246154      0.025631
IJzer [Fe]              55.85            Fe+2.tot     5.063103      0.000091
Magnesium [Mg]          24.31            Mg+2.tot   158.396296      0.006516
Calcium [Ca]            40.08            Ca+2.tot   363.474359      0.009069
Ammonium (als NH4)      18.04            NH4+.tot   446.129182       0.02473
Chloride                35.45             Cl-.tot   765.170490      0.021584
Bicarbonaat             61.02           HCO3-.tot  3776.996337      0.061898
Fosfaat (als PO4)       94.97           PO4-3.tot     8.366921      0.000088
Kalium [K]               39.1              K+.tot   255.116279      0.006525
Silicium [Si]           28.09              Si.tot   346.820448      0.012347

### Select data from data-set for further analysis
In the above code sections we have:
- imported the raw data;
- created plots to see the time-series for all available components;
- created a translation table with information necessary for calculating the molar concentrations and the component names used in pyOrchestra;
- identified which sampling-dates have the most chemical analyses available.

Now we are ready to extract the information from the selected date with most chemical analyses. Then we need to add a columns with the molar concentrations and orchestra component (or parameter) names.

In [11]:
# %%
# We use the date with most parameters for our first analysis
sel_idx = df_leachate['date'] == date_with_most_pars

df_work = df_leachate[sel_idx].copy()



# Export df_work to an Excel file so that we can 
# have a quick access to the parameters in it
# for setting up the translation from mg/l to moles/l 
# for the Orchestra input.
# %%
#df_work.to_excel('tmp/df_work_PP-11N.xlsx', index=False)

# %%
# Using the content from the file we now create a table
# with the parameters, their values and the conversion to moles/l.
# We will use this table to set up the translation from mg/l to moles/l
# for the Orchestra input.


# We can now use this conversion table 
# to convert the values in the df_work dataframe
# from mg/l to moles/l and to add a new column with Orchestra parameter names.

df_work['val_mol_l'] = df_work.apply(
    lambda row: 
        (row['val_mgl'] * 1e-3) / conversion_table.loc[row['cname'],'molar_mass'] 
        if row['cname'] in conversion_table.index else row['val_mgl'], axis=1)

df_work['orchestra_param'] = df_work.apply(
    lambda row: 
        conversion_table.loc[row['cname'], 'orchestra_parameter']
        if row['cname'] in conversion_table.index else row['cname'],axis=1
    )
    
#select temperatures and add 273.15 to convert to K
sel_temp = df_work['orchestra_param'] == 'T'
df_work.loc[sel_temp, 'val_mol_l'] += 273.15


In [12]:
print(df_work)
        

     compartment measpointname       date               cname      val_mgl  \
2195       BB11N        PP-11N 2025-12-11        Mangaan [Mn]     0.870000   
2196       BB11N        PP-11N 2025-12-11          IJzer [Fe]     1.000000   
2197       BB11N        PP-11N 2025-12-11        Natrium [Na]   440.000000   
2198       BB11N        PP-11N 2025-12-11   Sulfaat (als SO4)   329.862757   
2199       BB11N        PP-11N 2025-12-11                  pH     7.130000   
2200       BB11N        PP-11N 2025-12-11  Ammonium (als NH4)   150.373478   
2201       BB11N        PP-11N 2025-12-11             Sulfide     0.100000   
2202       BB11N        PP-11N 2025-12-11       Silicium [Si]    21.000000   
2203       BB11N        PP-11N 2025-12-11        Calcium [Ca]   460.000000   
2204       BB11N        PP-11N 2025-12-11            Chloride   507.000000   
2205       BB11N        PP-11N 2025-12-11   Fosfaat (als PO4)    11.250000   
2206       BB11N        PP-11N 2025-12-11         Bicarbonaat  3

### Add missing values to the data set
Although not required for this first analysis, when analysing samples with less analyses than 15, you need use the averaged value calculated above for the missing components. The following code section shows how to do this.

Once we have completed the processing of the sample we aim to analyse with pyOrchestra we save it to a spreadsheet in the local *tmp* directory. This allows us to easily set-up the Orchestra input files using the Orchestra GUI in the *orchestra2026.jar* file. For the current example this preparation already has been done, and if you do not have to add additional chemical components to the problem, you can run your own notebook using these Orchestra files as well.

To check the results we print the final processed data set.

In [13]:

# We need to add the missing parameters to df_work so that we have all the parameters in the conversion table in df_work.

# First we make cname the index of df_work so we can easily add the missing parameters
df_work.set_index('cname', inplace=True)

for cn in conversion_table.index:
    if cn not in df_work.index:
        df_work.loc[cn,'val_mol_l'] = conversion_table.loc[cn, 'avg_val_mol_l']
        df_work.loc[cn,'orchestra_param'] = conversion_table.loc[cn, 'orchestra_parameter']

print(df_work)

# %%
# Rewrite df_work to excel so we can copy the contents to the Orchestra input file.
df_work.to_excel('tmp/df_work_PP-11N.xlsx', index=False)
# %%


                   compartment measpointname       date      val_mgl  \
cname                                                                  
Mangaan [Mn]             BB11N        PP-11N 2025-12-11     0.870000   
IJzer [Fe]               BB11N        PP-11N 2025-12-11     1.000000   
Natrium [Na]             BB11N        PP-11N 2025-12-11   440.000000   
Sulfaat (als SO4)        BB11N        PP-11N 2025-12-11   329.862757   
pH                       BB11N        PP-11N 2025-12-11     7.130000   
Ammonium (als NH4)       BB11N        PP-11N 2025-12-11   150.373478   
Sulfide                  BB11N        PP-11N 2025-12-11     0.100000   
Silicium [Si]            BB11N        PP-11N 2025-12-11    21.000000   
Calcium [Ca]             BB11N        PP-11N 2025-12-11   460.000000   
Chloride                 BB11N        PP-11N 2025-12-11   507.000000   
Fosfaat (als PO4)        BB11N        PP-11N 2025-12-11    11.250000   
Bicarbonaat              BB11N        PP-11N 2025-12-11  3800.00

### Develop a strategy how you will analyse this sample with pyOrchestra...

The next steps which will mainly be carried out on Day 2 of this assignment, is to use the processed data you collected above in a series of pyOrchestra calculations. This requires you to revist the material from the previous weeks in this course:

1. What types of reactions do you expect to happen between the chemical components in the sample?
2. Which of these reactions take place in the water phase only?
3. Which of these reactions take place between the water phase and gas phase?
4. Which of these reactions take place between the water phase and solid phase?

In answering these questions, consider possible gases and minerals that may develop. The content from weeks 3.7 and 4.2 cover most of the material you need to figure this out.

Using the information you gathered you need to:
1. Choose which chemical compounds you use a your master species;
2. Check the chemistry.inp file with the Orchestra GUI so that it matches your choice of master species;
3. We need to answer 3 questions: 
    - what is the initial state of the sample?, 
    - what happens if it is equilibrated with the atmosphere?, 
    - and what were the conditions where it originated? 
4. What type of results do you expect to get from your pyOrchestra calculation, how will you interprete these results?
5. Start working on your report with all of the above information and think how your report should include your interpretation?